# 05. 취소 및 탑승 전환 분석

접수건이 취소 또는 실제 탑승으로 이어지는 비중을 시간대/요일 기준으로 분석한다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

df_request = pd.read_csv(
    'data/서울시설공단_장애인콜택시 접수일시_월주차_파생컬럼_20251231.csv',
    parse_dates=['접수일시', '접수일자']
)

df_request.info()
df_request.head()


In [ ]:
# CANCEL_REQUEST_DEMAND_ANALYSIS_JSON_CELL
# 6.7 취소 여부별 접수 수요 분석
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

if 'df_request' not in globals():
    df_request = pd.read_csv(
        'data/서울시설공단_장애인콜택시 접수일시_월주차_파생컬럼_20251231.csv'
    )

weekday_order = ['월요일', '화요일', '수요일', '목요일', '금요일', '토요일', '일요일']

df_cancel_analysis = df_request.copy()
df_cancel_analysis['취소여부'] = df_cancel_analysis['취소일시'].notna()
df_cancel_analysis['탑승여부'] = df_cancel_analysis['승차일시'].notna()
df_cancel_analysis['접수시간대_label'] = df_cancel_analysis['접수시간대'].map(
    lambda x: f'{int(x):02d}시' if pd.notna(x) else pd.NA
)

overall_cancel_summary = pd.DataFrame({
    '구분': ['전체 접수건수', '취소건수', '탑승건수', '미취소_미탑승건수'],
    '건수': [
        len(df_cancel_analysis),
        df_cancel_analysis['취소여부'].sum(),
        df_cancel_analysis['탑승여부'].sum(),
        ((~df_cancel_analysis['취소여부']) & (~df_cancel_analysis['탑승여부'])).sum(),
    ]
})
overall_cancel_summary['비중(%)'] = overall_cancel_summary['건수'] / len(df_cancel_analysis) * 100

time_cancel_summary = (
    df_cancel_analysis
    .dropna(subset=['접수시간대'])
    .groupby('접수시간대')
    .agg(
        전체접수건수=('접수일시', 'size'),
        취소건수=('취소여부', 'sum'),
        탑승건수=('탑승여부', 'sum')
    )
    .reindex(range(24), fill_value=0)
    .reset_index()
)
time_cancel_summary['접수시간대_label'] = time_cancel_summary['접수시간대'].map(lambda x: f'{int(x):02d}시')
time_cancel_summary['취소비중(%)'] = time_cancel_summary['취소건수'] / time_cancel_summary['전체접수건수'] * 100
time_cancel_summary['탑승전환비중(%)'] = time_cancel_summary['탑승건수'] / time_cancel_summary['전체접수건수'] * 100

day_cancel_summary = (
    df_cancel_analysis
    .dropna(subset=['접수요일'])
    .groupby('접수요일')
    .agg(
        전체접수건수=('접수일시', 'size'),
        취소건수=('취소여부', 'sum'),
        탑승건수=('탑승여부', 'sum')
    )
    .reindex(weekday_order)
    .reset_index()
)
day_cancel_summary['취소비중(%)'] = day_cancel_summary['취소건수'] / day_cancel_summary['전체접수건수'] * 100
day_cancel_summary['탑승전환비중(%)'] = day_cancel_summary['탑승건수'] / day_cancel_summary['전체접수건수'] * 100

weekday_hour_cancel_summary = (
    df_cancel_analysis
    .dropna(subset=['접수요일', '접수시간대'])
    .groupby(['접수요일', '접수시간대'])
    .agg(
        전체접수건수=('접수일시', 'size'),
        취소건수=('취소여부', 'sum'),
        탑승건수=('탑승여부', 'sum')
    )
    .reset_index()
)
weekday_hour_cancel_summary['취소비중(%)'] = weekday_hour_cancel_summary['취소건수'] / weekday_hour_cancel_summary['전체접수건수'] * 100
weekday_hour_cancel_summary['탑승전환비중(%)'] = weekday_hour_cancel_summary['탑승건수'] / weekday_hour_cancel_summary['전체접수건수'] * 100
weekday_hour_cancel_summary['접수시간대_label'] = weekday_hour_cancel_summary['접수시간대'].map(lambda x: f'{int(x):02d}시')

high_request_low_boarding = (
    weekday_hour_cancel_summary[
        weekday_hour_cancel_summary['전체접수건수'] >= weekday_hour_cancel_summary['전체접수건수'].quantile(0.75)
    ]
    .sort_values(['탑승전환비중(%)', '전체접수건수'], ascending=[True, False])
    .reset_index(drop=True)
)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

axes[0, 0].bar(
    overall_cancel_summary['구분'],
    overall_cancel_summary['건수'],
    color=['#4c78a8', '#e45756', '#54a24b', '#b279a2']
)
axes[0, 0].set_title('전체 접수건 중 취소/탑승 비중')
axes[0, 0].set_ylabel('건수')
axes[0, 0].tick_params(axis='x', rotation=20)
axes[0, 0].grid(axis='y', alpha=0.3)
for index, row in overall_cancel_summary.iterrows():
    label_text = f"{row['건수']:,.0f}건 ({row['비중(%)']:.1f}%)"
    axes[0, 0].text(index, row['건수'], label_text, ha='center', va='bottom', fontsize=9)

axes[0, 1].plot(time_cancel_summary['접수시간대_label'], time_cancel_summary['취소비중(%)'], marker='o', color='#e45756', label='취소비중')
axes[0, 1].plot(time_cancel_summary['접수시간대_label'], time_cancel_summary['탑승전환비중(%)'], marker='o', color='#54a24b', label='탑승전환비중')
axes[0, 1].set_title('시간대별 취소비중과 탑승전환비중')
axes[0, 1].set_xlabel('접수시간대')
axes[0, 1].set_ylabel('비중(%)')
axes[0, 1].tick_params(axis='x', rotation=0)
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

axes[1, 0].bar(day_cancel_summary['접수요일'], day_cancel_summary['취소비중(%)'], color='#e45756')
axes[1, 0].set_title('요일별 취소비중')
axes[1, 0].set_xlabel('접수요일')
axes[1, 0].set_ylabel('취소비중(%)')
axes[1, 0].tick_params(axis='x', rotation=0)
axes[1, 0].grid(axis='y', alpha=0.3)
for index, value in enumerate(day_cancel_summary['취소비중(%)']):
    axes[1, 0].text(index, value, f' {value:.1f}%', ha='center', va='bottom', fontsize=9)

cancel_rate_pivot = (
    weekday_hour_cancel_summary
    .pivot_table(index='접수요일', columns='접수시간대_label', values='취소비중(%)', aggfunc='mean')
    .reindex(weekday_order)
)
cancel_rate_pivot = cancel_rate_pivot[[f'{hour:02d}시' for hour in range(24) if f'{hour:02d}시' in cancel_rate_pivot.columns]]

sns.heatmap(
    cancel_rate_pivot,
    ax=axes[1, 1],
    cmap='YlOrRd',
    linewidths=0.4,
    cbar_kws={'label': '취소비중(%)'}
)
axes[1, 1].set_title('요일 X 시간대별 취소비중 히트맵')
axes[1, 1].set_xlabel('접수시간대')
axes[1, 1].set_ylabel('접수요일')
axes[1, 1].tick_params(axis='x', rotation=0)
axes[1, 1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

print('전체 취소/탑승 요약')
display(overall_cancel_summary)

print('시간대별 취소비중 상위 10')
display(time_cancel_summary.sort_values('취소비중(%)', ascending=False).head(10))

print('요일별 취소비중')
display(day_cancel_summary)

print('접수 수요는 많지만 탑승전환비중이 낮은 요일 X 시간대 후보')
display(high_request_low_boarding.head(20))

